# TikTok Creator Intelligence

This notebook has two sections:

**Section A** — Install packages and launch the Streamlit web app 
**Section B** — Run the TikTok data scraper (separate, run after you have your msToken)

---

## Section A — Streamlit Web App

### Step 1 — Install all packages

This installs everything the app needs:
- `streamlit` — the web framework that runs your app in a browser
- `pandas` — for reading and processing the CSV data
- `plotly` — for the charts and graphs
- `nltk` — Natural Language Toolkit, provides the VADER sentiment analyser
- `scikit-learn` — provides TF-IDF for keyword extraction
- `openpyxl` — lets the app read Excel files
- `vaderSentiment` — the sentiment scoring library

Run this cell once. The `!` means 'run in terminal, not Python'.

In [ ]:
!python -m pip install streamlit pandas plotly nltk scikit-learn openpyxl vaderSentiment

### Step 2 — Download the VADER language data

VADER (the sentiment analyser) needs to download its word list once before it can work.
This cell does that download. Run it once.

In [ ]:
import nltk
nltk.download('vader_lexicon')
print('VADER lexicon downloaded successfully.')

### Step 3 — Launch the web app

This starts the Streamlit server and automatically opens the app in your browser.

**What to expect:**
- A browser tab opens at `http://localhost:8501`
- The app starts on the Login page
- Register a new account, then log in
- The Upload Data page loads with demo data already analysed
- Navigate using the sidebar

> To stop the app: click the cell output area and press **I, I** (interrupt kernel), or click the stop button.

In [ ]:
!python -m streamlit run app/main.py

---

## Section B — TikTok Data Scraper

Run these cells to collect real comment data from your TikTok account.
Complete Section A first so all packages are installed.

### Step 4 — Install scraper packages

This installs the TikTokApi library and Playwright browser automation.
Run once.

In [ ]:
!python -m pip install TikTokApi==6.5.2 playwright>=1.40.0

### Step 5 — Download the Chromium browser

Playwright needs a real browser to open TikTok. This downloads a copy of Chromium.
Run once.

In [ ]:
!python -m playwright install chromium

### Step 6 — Run the scraper

**Before running:** If TikTok blocked your login attempts, skip the browser login.
Instead, get your `msToken` manually:
1. Open TikTok in Chrome and log in normally
2. Press F12 → Application tab → Cookies → tiktok.com
3. Find `msToken`, copy the value
4. Create a file called `session.json` in the project folder with:
   ```json
   {"ms_token": "PASTE_YOUR_TOKEN_HERE", "saved_at": "2026-01-01", "username": "ichbinnelo"}
   ```

Then run the cell below. It will collect all your videos and comments into `data/`.

In [ ]:
!python scrape.py

### Step 7 — Check scraped files

In [ ]:
import os
from pathlib import Path

data_folder = Path('./data')
files = [f for f in data_folder.iterdir() if f.suffix == '.csv']

if files:
    print('CSV files in data/ folder:')
    for f in sorted(files):
        size_kb = f.stat().st_size / 1024
        print(f'  {f.name}  ({size_kb:.1f} KB)')
else:
    print('No CSV files found yet. Run the scraper in Step 6 first.')

### Step 8 — Preview scraped data

In [ ]:
import pandas as pd
from pathlib import Path

data_folder = Path('./data')

video_files   = sorted(data_folder.glob('videos_*.csv'),   reverse=True)
comment_files = sorted(data_folder.glob('comments_*.csv'), reverse=True)

if not video_files:
    print('No scraped data found yet. Run the scraper in Step 6 first.')
else:
    videos_df   = pd.read_csv(video_files[0],   encoding='utf-8-sig')
    comments_df = pd.read_csv(comment_files[0], encoding='utf-8-sig')

    print(f'Videos:   {len(videos_df)} rows  —  {video_files[0].name}')
    print(f'Comments: {len(comments_df)} rows  —  {comment_files[0].name}')
    print()
    print('--- Videos preview ---')
    display(videos_df.head())
    print('--- Comments preview ---')
    display(comments_df.head())

---

## Section C — Exploratory Data Analysis (Deliverable 5)

This section covers the **Exploratory Showcase** required for the Data Strategy deliverable.  
All analysis runs on the real scraped data from `@ichbinnelo`.

Sections:
- **C1** — Dataset statistics
- **C2** — Pre-processing: before vs after
- **C3** — Comment length distribution
- **C4** — Top 20 most frequent terms
- **C5** — Video type distribution
- **C6** — Language breakdown
- **C7** — NLP test: VADER sentiment on sample comments

In [ ]:
# Install EDA dependencies (run once)
!python -m pip install matplotlib vaderSentiment langdetect deep-translator -q
import nltk
nltk.download('vader_lexicon', quiet=True)
nltk.download('stopwords',    quiet=True)
print('All EDA dependencies ready.')

### C1 — Dataset Statistics

In [ ]:
import pandas as pd
from pathlib import Path

# Load datasets
videos   = pd.read_csv('data/videos_merged.csv',            encoding='utf-8-sig')
comments = pd.read_csv('data/comments_20260526_233400.csv', encoding='utf-8-sig')

# Comment length in words
comments['word_count'] = comments['comment_text'].dropna().apply(lambda x: len(str(x).split()))

# Date range
comments['comment_date'] = pd.to_datetime(comments['comment_date'], errors='coerce', utc=True)
date_min = comments['comment_date'].min()
date_max = comments['comment_date'].max()

# Comments per video
cpm = comments.groupby('video_id').size()

print('=' * 50)
print('  DATASET SUMMARY')
print('=' * 50)
print(f'  Total videos            : {len(videos)}')
print(f'  Videos with metrics     : {(videos["view_count"] > 0).sum()}')
print(f'  Video categories        : {videos["video_type"].nunique()}')
print()
print(f'  Total comments          : {len(comments)}')
print(f'  Unique videos commented : {comments["video_id"].nunique()}')
print(f'  Unique commenters       : {comments["author_username"].nunique()}')
print()
print(f'  Comments per video')
print(f'    Min    : {cpm.min()}')
print(f'    Max    : {cpm.max()}')
print(f'    Mean   : {cpm.mean():.1f}')
print(f'    Median : {cpm.median():.0f}')
print()
print(f'  Comment length (words)')
print(f'    Min    : {comments["word_count"].min()}')
print(f'    Max    : {comments["word_count"].max()}')
print(f'    Mean   : {comments["word_count"].mean():.1f}')
print()
print(f'  Content date range      : {date_min.date() if pd.notna(date_min) else "n/a"}  →  {date_max.date() if pd.notna(date_max) else "n/a"}')
print(f'  Unique vocabulary       : {len(set(" ".join(comments["comment_text"].dropna()).lower().split()))} tokens')
print('=' * 50)

### C2 — Pre-processing: Before vs After

Demonstrates the full cleaning pipeline on a representative raw comment.

In [ ]:
import re

def clean_text(text):
    """Mirrors nlp/preprocessor.py — used here for the before/after demo."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'http\S+|www\S+',          '', text)   # URLs
    text = re.sub(r'@\w+',                     '', text)   # @mentions
    text = re.sub(r'#\w+',                     '', text)   # #hashtags
    text = re.sub(r'[^\w\s]',                  '', text)   # punctuation
    text = re.sub(r'\d+',                      '', text)   # numbers
    text = re.sub(r'[^\x00-\x7F]+',            '', text)   # emojis / non-ASCII
    text = re.sub(r'\s+',                      ' ', text)  # extra whitespace
    tokens = [t for t in text.split() if len(t) >= 3]      # drop short tokens
    return ' '.join(tokens)

# Representative examples — one per cleaning challenge
examples = [
    "omg I love this so much!!! 🥰🥰 #foodtok #fyp check https://t.co/abc123",
    "Ich wünsche dir einen wunderschönen Tag ❤️",          # was German, now translated
    "@ichbinnelo please do more cooking videos!! 😍",
    "Die von Subway 😭😍",                                  # short translated German
    "this and starting my seedlings #CapCut #foodtok",
]

print(f'{"RAW INPUT":<55}  →  CLEANED OUTPUT')
print('-' * 90)
for raw in examples:
    cleaned = clean_text(raw)
    print(f'{raw[:55]:<55}  →  {cleaned if cleaned else "(empty — too short after cleaning)"}')


### C3 — Comment Length Distribution

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

word_counts = comments['comment_text'].dropna().apply(lambda x: len(str(x).split()))

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(word_counts[word_counts <= 40], bins=40, color='#a78bfa', edgecolor='white', linewidth=0.5)

ax.set_title('Comment Length Distribution  (@ichbinnelo)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Word count per comment', fontsize=11)
ax.set_ylabel('Number of comments',     fontsize=11)
ax.axvline(word_counts.mean(),  color='#f472b6', linestyle='--', linewidth=1.5, label=f'Mean  {word_counts.mean():.1f} words')
ax.axvline(word_counts.median(),color='#34d399', linestyle='--', linewidth=1.5, label=f'Median {word_counts.median():.0f} words')
ax.legend(fontsize=10)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('data/eda_comment_length.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Most comments are short: {(word_counts <= 5).sum()} out of {len(word_counts)} ({(word_counts<=5).mean()*100:.0f}%) are 5 words or fewer.')
print(f'Longest comment: {word_counts.max()} words.')


### C4 — Top 20 Most Frequent Terms

After stopword removal — shows the dominant vocabulary in the comment corpus.

In [ ]:
from collections import Counter
from nltk.corpus import stopwords
import matplotlib.pyplot as plt

STOPWORDS = set(stopwords.words('english')) | {
    'like', 'just', 'get', 'got', 'one', 'also', 'going', 'know', 'want',
    'lol', 'omg', 'iam', 'im', 'ur', 'u', 'dont', 'cant', 'its', 'thats',
    'yes', 'hey', 'hi', 'ok', 'okay', 'done', 'ily', 'haha', 'pls', 'plz',
}

all_tokens = []
for text in comments['comment_text'].dropna():
    cleaned = clean_text(str(text))
    tokens  = [t for t in cleaned.split() if t not in STOPWORDS and len(t) >= 3]
    all_tokens.extend(tokens)

top20 = Counter(all_tokens).most_common(20)
words, counts = zip(*top20)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(list(reversed(words)), list(reversed(counts)), color='#c084fc', edgecolor='white')
ax.set_title('Top 20 Most Frequent Terms in Comments  (@ichbinnelo)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Frequency', fontsize=11)
ax.spines[['top','right','left']].set_visible(False)
ax.tick_params(left=False)
for bar, count in zip(bars, reversed(counts)):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            str(count), va='center', fontsize=9, color='#555')

plt.tight_layout()
plt.savefig('data/eda_top_terms.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 20 terms and counts:')
for word, count in top20:
    print(f'  {word:<20} {count}')


### C5 — Video Type Distribution

In [ ]:
import matplotlib.pyplot as plt

type_counts = videos['video_type'].value_counts()

COLORS = {
    'Lifestyle & Vlog':     '#a78bfa',
    'Unknown':              '#d1d5db',
    'Personal Finance':     '#34d399',
    'Student Life':         '#60a5fa',
    'Food & Cooking':       '#fb923c',
    'Engagement / Growth':  '#f472b6',
    'Books & Reading':      '#fbbf24',
    'Hair & Beauty':        '#e879f9',
    'Motivation':           '#4ade80',
    'Travel & Places':      '#38bdf8',
}
colors = [COLORS.get(t, '#9ca3af') for t in type_counts.index]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(type_counts.index[::-1], type_counts.values[::-1],
               color=colors[::-1], edgecolor='white')

ax.set_title('Video Distribution by Content Type  (@ichbinnelo)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Number of videos', fontsize=11)
ax.spines[['top','right','left']].set_visible(False)
ax.tick_params(left=False)
for bar, val in zip(bars, type_counts.values[::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=10, color='#333')

plt.tight_layout()
plt.savefig('data/eda_video_types.png', dpi=150, bbox_inches='tight')
plt.show()

print('Video type counts:')
for vtype, count in type_counts.items():
    pct = count / len(videos) * 100
    print(f'  {vtype:<25} {count:>3}  ({pct:.0f}%)')


### C6 — Language Breakdown

Shows how many comments were detected as non-English and required translation.

In [ ]:
import matplotlib.pyplot as plt

lang_counts = comments['language'].value_counts()

# Group minor languages into "Other"
top_langs  = lang_counts[lang_counts >= 5]
other_count = lang_counts[lang_counts < 5].sum()
if other_count > 0:
    top_langs = top_langs._append(pd.Series({'other': other_count}))

english_count     = lang_counts.get('en', 0)
non_english_count = len(comments) - english_count

print(f'Language detection results:')
print(f'  Detected as English     : {english_count} ({english_count/len(comments)*100:.0f}%)')
print(f'  Detected as non-English : {non_english_count} ({non_english_count/len(comments)*100:.0f}%)')
print(f'  (non-English comments were translated to English before NLP analysis)')
print()

# Pie chart — English vs non-English
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.pie(
    [english_count, non_english_count],
    labels=['English', 'Translated'],
    colors=['#a78bfa', '#f472b6'],
    autopct='%1.0f%%',
    startangle=90,
    textprops={'fontsize': 12},
)
ax1.set_title('English vs Translated Comments', fontsize=12, fontweight='bold')

# Bar chart — top detected languages
top_display = lang_counts.head(10)
ax2.barh(top_display.index[::-1], top_display.values[::-1], color='#c084fc', edgecolor='white')
ax2.set_title('Top 10 Detected Languages', fontsize=12, fontweight='bold')
ax2.set_xlabel('Comment count', fontsize=10)
ax2.spines[['top','right','left']].set_visible(False)
ax2.tick_params(left=False)

plt.tight_layout()
plt.savefig('data/eda_languages.png', dpi=150, bbox_inches='tight')
plt.show()

print('Note: short single-word comments (e.g. "yummy", "wow") are often misclassified')
print('by the language detector. The underlying text is English and translation had no effect.')


### C7 — NLP Test: VADER Sentiment on Sample Comments

Runs the full VADER sentiment pipeline on 10 real comments and prints the compound score and classification for each.  
This is the same logic used in `nlp/sentiment.py`.

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def classify_sentiment(compound):
    if compound >= 0.05:  return 'POSITIVE'
    if compound <= -0.05: return 'NEGATIVE'
    return 'NEUTRAL'

# Hand-pick 10 representative comments spanning the sentiment spectrum
sample_comments = [
    "omg I love this so much it looks amazing",
    "this is my favorite type of content please keep making these",
    "I wish you a wonderful day",
    "berlin is nice for tourists but not for residents",
    "what was the first thing in the video this one with strawberries looked delicious",
    "it is not such a bad winter day is it",
    "Please do it because I am not that famous yet",
    "I have been trying to pay off my debt for years this really helped",
    "I add back everyone who adds me",
    "My condolences",
]

print(f'{"COMMENT":<55}  {"SCORE":>7}  LABEL')
print('-' * 80)
for comment in sample_comments:
    scores   = analyzer.polarity_scores(comment)
    compound = scores['compound']
    label    = classify_sentiment(compound)
    marker   = {'POSITIVE': '+', 'NEGATIVE': '-', 'NEUTRAL': ' '}[label]
    print(f'{comment[:55]:<55}  {compound:>+.3f}  [{marker}] {label}')

print()
# Run on the full dataset and show aggregate
all_scores = comments['comment_text'].dropna().apply(
    lambda x: classify_sentiment(analyzer.polarity_scores(x)['compound'])
)
dist = all_scores.value_counts()
total = len(all_scores)
print('Full corpus sentiment distribution:')
for label, count in dist.items():
    bar = '#' * int(count / total * 40)
    print(f'  {label:<10} {count:>5}  ({count/total*100:4.1f}%)  {bar}')
